# Streaming

Receive responses in real-time via Server-Sent Events (SSE) instead of waiting for the full response. This notebook demonstrates how to enable streaming and parse SSE events from the Jockey API.

In [ ]:
import json
import os
import time

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## When You Need Streaming

- Building a chat-like UI where tokens appear as they are generated
- Processing long responses where you want to show progress
- Reducing perceived latency for end users

## Enable Streaming

To enable streaming, set `"stream": True` in the JSON body **and** pass `stream=True` to `requests.post()`. Both are required.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "stream": True,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Describe what happens in these videos",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
    stream=True,
)

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type', 'N/A')}")

## Parsing SSE Events

The response is a stream of Server-Sent Events. Each event is a line prefixed with `data: ` followed by a JSON payload. The stream ends with `data: [DONE]`.

In [ ]:
def parse_sse_stream(response: requests.Response) -> list[dict]:
    """Parse an SSE response stream and return all events.

    Args:
        response: A streaming requests.Response object.

    Returns:
        A list of parsed event dictionaries.
    """
    events = []
    for line in response.iter_lines():
        if line:
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                data = decoded[6:]  # Strip "data: " prefix
                if data == "[DONE]":
                    print("\n--- Stream complete ---")
                    break
                event = json.loads(data)
                events.append(event)
                print(event)
    return events

In [ ]:
# Make a fresh streaming request and parse the events
streaming_response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "stream": True,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Describe what happens in these videos",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
    stream=True,
)

events = parse_sse_stream(streaming_response)
print(f"\nTotal events received: {len(events)}")

## Streaming with Text Accumulation

In a real application you typically want to accumulate the text content as it streams in. Here is a helper that collects the full text from SSE events.

In [ ]:
def stream_text(response: requests.Response) -> str:
    """Stream SSE events and accumulate the full text response.

    Args:
        response: A streaming requests.Response object.

    Returns:
        The accumulated text content.
    """
    full_text = ""
    for line in response.iter_lines():
        if line:
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                data = decoded[6:]
                if data == "[DONE]":
                    break
                event = json.loads(data)
                # Extract text content from the event if present
                if "output" in event:
                    for output in event["output"]:
                        if output.get("type") == "message":
                            for content in output.get("content", []):
                                chunk = content.get("text", "")
                                full_text += chunk
                                print(chunk, end="", flush=True)
    print()  # Final newline
    return full_text

## Common Pitfalls

- **Set `stream=True` in requests library** -- both in the JSON body AND the `requests.post()` call
- **Handle the `[DONE]` signal** -- it marks the end of the stream
- **Reconnection** -- if the connection drops, start a new request (SSE reconnection is not automatic)

## Next Steps

- [Structured Output](./structured_output.ipynb) -- Force Jockey to return typed JSON matching a schema you define
- [Multi-Turn Sessions](./multi_turn_sessions.ipynb) -- Continue conversations across multiple requests
- [Error Handling](./error_handling.ipynb) -- Retry strategies, polling helpers, and common error patterns
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)